In [ ]:
import numpy as np
import pandas as pd
from numpy.random import default_rng
from scipy.linalg import sqrtm
import matplotlib.pyplot as plt
from io import BytesIO

# -----------------------------
# Configuration
# -----------------------------
N_POINTS = 101
DEPTH    = 33
N_RANDOM = 24
EPS      = 1e-6
rng = default_rng(42)

# -----------------------------
# Linear algebra + channels
# -----------------------------
I2 = np.eye(2, dtype=np.complex128)
X  = np.array([[0, 1],[1, 0]], dtype=np.complex128)
Y  = np.array([[0, -1j],[1j, 0]], dtype=np.complex128)
Z  = np.array([[1, 0],[0, -1]], dtype=np.complex128)


def pure_to_rho(psi):
    psi = np.asarray(psi, dtype=np.complex128).reshape(2,1)
    psi = psi / np.linalg.norm(psi)
    return psi @ psi.conj().T

def haar_random_pure_qubit(n: int):
    states = []
    for _ in range(n):
        z = rng.normal(size=2) + 1j*rng.normal(size=2)
        psi = z / np.linalg.norm(z)
        states.append(pure_to_rho(psi))
    return states

def fidelity(rho, sigma):
    sr = sqrtm(rho)
    inner = sr @ sigma @ sr
    inner = (inner + inner.conj().T) / 2.0
    root  = sqrtm(inner)
    val   = np.real(np.trace(root))**2
    return float(np.clip(val, 0.0, 1.0))

def bures_distance(rho, sigma):
    F = fidelity(rho, sigma)
    return float(np.sqrt(max(0.0, 2.0*(1.0 - np.sqrt(F)))))

def apply_kraus(rho, Ks):
    out = np.zeros((2,2), dtype=np.complex128)
    for K in Ks:
        out += K @ rho @ K.conj().T
    out = (out + out.conj().T) / 2.0
    tr  = np.real(np.trace(out))
    return out / tr if tr != 0 else I2/2

# --- Channels ---

def dephasing_kraus(p):
    K0 = np.sqrt(1 - p) * I2
    K1 = np.sqrt(p) * np.array([[1,0],[0,0]], dtype=np.complex128)
    K2 = np.sqrt(p) * np.array([[0,0],[0,1]], dtype=np.complex128)
    return [K0, K1, K2]

def depolarizing_kraus(p):
    return [np.sqrt(1.0 - p)*I2,
            np.sqrt(p/3.0)*X,
            np.sqrt(p/3.0)*Y,
            np.sqrt(p/3.0)*Z]

def amplitude_damping_kraus(g):
    K0 = np.array([[1,0],[0,np.sqrt(1-g)]], dtype=np.complex128)
    K1 = np.array([[0,np.sqrt(g)],[0,0]], dtype=np.complex128)
    return [K0, K1]

# -----------------------------
# Trajectories & indices
# -----------------------------
def trajectory(rho0, kraus_fn, param, depth):
    traj = [rho0]
    Ks = kraus_fn(param)
    rho = rho0
    for _ in range(depth):
        rho = apply_kraus(rho, Ks)
        traj.append(rho)
    return traj

def trajectory_length_bures(traj):
    return sum(bures_distance(traj[i], traj[i+1]) for i in range(len(traj)-1))

def gdi(traj, eps=1e-9):
    LB = trajectory_length_bures(traj)
    DB = bures_distance(traj[0], traj[-1])
    return float(LB / max(DB, eps))

def gac_mean(traj):
    T = traj[-1]; vals = []
    for i in range(len(traj)-1):
        A, B = traj[i], traj[i+1]
        dAB, dAT, dBT = bures_distance(A,B), bures_distance(A,T), bures_distance(B,T)
        denom = 2.0 * dAB * dAT
        if denom <= 1e-12:
            continue
        cos_th = (dAB**2 + dAT**2 - dBT**2) / denom
        vals.append(np.clip(np.real(cos_th), -1.0, 1.0))
    return float(np.mean(vals)) if vals else 1.0

def sample_states(n_random: int):
    return haar_random_pure_qubit(n_random)

# -----------------------------
# Sweep runner
# -----------------------------
def run_sweep_means(channel_name, kraus_fn, grid, depth, n_random):
    rows = []
    states = sample_states(n_random)
    for param in grid:
        Fs, DBs, GDIs, GACs = [], [], [], []
        for rho0 in states:
            traj = trajectory(rho0, kraus_fn, float(param), depth)
            Fs.append(fidelity(traj[0], traj[-1]))
            DBs.append(bures_distance(traj[0], traj[-1]))
            GDIs.append(gdi(traj))
            GACs.append(gac_mean(traj))
        rows.append({
            "channel": channel_name,
            "param": float(param),
            "F":   float(np.mean(Fs)),
            "DB":  float(np.mean(DBs)),
            "GDI": float(np.mean(GDIs)),
            "GAC": float(np.mean(GACs)),
        })
    return pd.DataFrame(rows)

# -----------------------------
# Grid & sweeps
# -----------------------------
grid_01 = np.linspace(0.0 + EPS, 1.0 - EPS, N_POINTS)

df_deph = run_sweep_means("dephasing",        dephasing_kraus,         grid_01, DEPTH, N_RANDOM)
df_depo = run_sweep_means("depolarizing",     depolarizing_kraus,      grid_01, DEPTH, N_RANDOM)
df_ad   = run_sweep_means("amplitude_damping",amplitude_damping_kraus, grid_01, DEPTH, N_RANDOM)

# -----------------------------
# Plot helpers (inline + JPG in memory)
# -----------------------------
def plot_channel_curves(df, title, xlabel):
    d = df.sort_values("param")
    plt.figure()
    plt.plot(d["param"], d["F"],   label="Fidelity")
    plt.plot(d["param"], d["DB"],  label="Bures distance")
    plt.plot(d["param"], d["GDI"], label="GDI")
    plt.plot(d["param"], d["GAC"], label="GAC")
    plt.xlabel(xlabel)
    plt.ylabel("Index value")
    plt.title(title)
    plt.ylim(0,2)
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()

    # Show in Colab
    plt.show()

    # Save as JPG in memory
    buf = BytesIO()
    plt.savefig(buf, dpi=300, format="jpg")
    buf.seek(0)
    return buf

img_deph = plot_channel_curves(df_deph, "Dephasing channel",        "Noise parameter p")
img_depo = plot_channel_curves(df_depo, "Depolarizing channel",     "Noise parameter p")
img_ad   = plot_channel_curves(df_ad,   "Amplitude damping channel","Damping parameter γ")

# -----------------------------
# Panel (inline + JPG buffer)
# -----------------------------
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), sharey=True)

# Dephasing
d = df_deph.sort_values("param")
axes[0].plot(d["param"], d["F"],   label="Fidelity")
axes[0].plot(d["param"], d["DB"],  label="Bures distance")
axes[0].plot(d["param"], d["GDI"], label="GDI")
axes[0].plot(d["param"], d["GAC"], label="GAC")
axes[0].set_title("Dephasing")
axes[0].set_xlabel("Noise parameter p")
axes[0].set_ylabel("Index value")
axes[0].set_ylim(0,2)
axes[0].grid(alpha=0.3)

# Depolarizing
d = df_depo.sort_values("param")
axes[1].plot(d["param"], d["F"])
axes[1].plot(d["param"], d["DB"])
axes[1].plot(d["param"], d["GDI"])
axes[1].plot(d["param"], d["GAC"])
axes[1].set_title("Depolarizing")
axes[1].set_xlabel("Noise parameter p")
axes[1].set_ylim(0,2)
axes[1].grid(alpha=0.3)

# Amplitude damping
d = df_ad.sort_values("param")
axes[2].plot(d["param"], d["F"])
axes[2].plot(d["param"], d["DB"])
axes[2].plot(d["param"], d["GDI"])
axes[2].plot(d["param"], d["GAC"])
axes[2].set_title("Amplitude damping")
axes[2].set_xlabel("Damping parameter γ")
axes[2].set_ylim(0,2)
axes[2].grid(alpha=0.3)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=4, frameon=False, bbox_to_anchor=(0.5, -0.05))

plt.tight_layout()
plt.subplots_adjust(bottom=0.22)

# Show panel
plt.show()

# Save in-memory JPG
panel_buf = BytesIO()
fig.savefig(panel_buf, dpi=300, format="jpg")
panel_buf.seek(0)
